# Notebook T4.2 — Pipeline de Extracción Estructurada
## Patrón: "Del texto caótico a datos limpios"

**Referencia en la guía metodológica:** PASO 5 → PATRÓN B  
**Nivel:** Principiante en GenAI

---

### ¿Qué problema resuelve este notebook?

En el mundo real, una gran parte de los datos llega en **formato no estructurado**: 
emails, tickets de soporte, formularios de texto libre, comentarios, etc.

El ML clásico no puede procesar texto directamente — necesita números o categorías fijas.  
El LLM actúa como **parser inteligente**: lee el texto y extrae exactamente los campos que necesitas 
en formato JSON, que luego puedes usar con cualquier herramienta de ML o análisis.

**Caso de uso:** Sistema de ticketing de soporte al cliente.  
Cada ticket llega como texto libre. Necesitamos extraer: categoría, urgencia, sentimiento, etc.

---

### Flujo completo del pipeline

```
Texto libre (email/ticket)
         ↓
  Prompt de extracción
         ↓
  LLM → JSON estructurado
         ↓
  DataFrame de pandas
         ↓
  Análisis ML / Priorización / Respuesta automática
```

---

### Conexión con la guía metodológica

- **Paso 1:** Datos de entrada = texto libre; salida = acciones sobre tickets  
- **Paso 3:** Prompt de extracción con formato JSON explícito (temperatura = 0)  
- **Paso 4:** El parser JSON maneja el output del LLM  
- **Paso 5:** PATRÓN B — LLM extrae → JSON → ML downstream  
- **Paso 8:** Función `extraer_datos()` encapsula el pipeline de extracción

## PASO 2 — Instalación y Configuración

### ¿Por qué temperatura = 0 en extracción?

En extracción de datos la **determinismo es crítico**: queremos que el LLM 
siempre devuelva el mismo JSON para el mismo texto, no respuestas creativas.  
`temperature=0` hace al LLM lo más determinista posible.

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Lee el archivo .env si existe
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")

# temperature=0 → máxima determinismo para extracción de datos
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

print("✅ LLM configurado con temperature=0 para extracción determinista")

## PASO 4 — Dataset: Tickets de Soporte en Texto Libre

### ¿Por qué este formato de datos es un problema para ML clásico?

Estos textos son **datos no estructurados**. Tienen toda la información que necesitas 
(urgencia, categoría, sentimiento), pero en forma de prosa humana, no en columnas.

Un modelo ML no puede leer "Estoy muy frustrada" y entender que el sentimiento es negativo.  
Necesita un número como -0.9 o una etiqueta como "muy_negativo".

El LLM resolverá esta brecha en el PASO 5.

In [ ]:
tickets_texto = [
    """Hola, soy María González. Llevo 3 días sin poder acceder a mi cuenta premium.
    Intenté restablecer la contraseña pero el email no llega. Estoy muy frustrada,
    ya pagué este mes y no puedo usar el servicio. Necesito solución urgente.""",

    """Buenos días. Mi nombre es Carlos López. Quería consultar si es posible
    hacer upgrade a plan enterprise para mi empresa. Somos 15 personas y el plan
    actual nos queda chico. ¿Tienen descuentos por volumen?""",

    """Hola! Soy Ana Martínez. Encontré un bug en la aplicación móvil:
    cuando intento exportar a PDF en iOS se cierra la app. Pasa siempre
    en iPhone 14 con iOS 17. Adjunto el log de error.""",

    """Buenas tardes. Soy Roberto Silva. Estoy pensando en cancelar mi suscripción
    porque el precio subió mucho y no uso todas las features. ¿Tienen algún plan
    más económico o descuento por lealtad? Llevo 2 años con ustedes.""",

    """Hola, Pedro Ramírez aquí. El cobro de este mes fue incorrecto,
    me cobraron $150 pero debería ser $80 según mi plan básico.
    Necesito que revisen y hagan el reintegro lo antes posible.""",

    """Buenas! Laura Fernández. Me encanta el producto, funciona genial.
    Solo quería sugerir que agreguen integración con Notion porque lo uso
    mucho para trabajar. Sería increíble tenerlo todo conectado.""",
]

print(f"Total de tickets a procesar: {len(tickets_texto)}")
print(f"\nEjemplo del ticket 1 (texto libre, sin estructura):")
print("-" * 50)
print(tickets_texto[0])

## PASO 5 — Extracción Estructurada con LLM

### El prompt de extracción: la clave está en el formato JSON

El prompt le pide al LLM que actúe como un **parser de información**.  
Puntos críticos del diseño:

1. **"Responde ÚNICAMENTE con el JSON"**: sin esta instrucción, el LLM añadiría texto extra 
   que rompería el `json.loads()` al intentar parsear
2. **Categorías fijas**: `"uno de: bug_tecnico | consulta_comercial | ..."` — el LLM elige entre opciones, 
   no inventa categorías nuevas (igual que un label en ML)
3. **Llaves dobles `{{ }}`**: en los templates de LangChain, `{variable}` es una variable, 
   `{{literal}}` es una llave literal que aparecerá en el prompt

### La función `extraer_datos()` y el parser JSON

```
ticket_texto (str)
    → chain.invoke()    # llama al LLM con el prompt
    → raw (str)         # respuesta cruda del LLM (puede tener ```json ... ```)
    → limpieza          # eliminar markdown backticks
    → json.loads()      # convertir string → dict de Python
    → dict con los campos extraídos
```

In [ ]:
prompt_extraccion = ChatPromptTemplate.from_template("""
Analiza el siguiente ticket de soporte y extrae la información en formato JSON.
Responde ÚNICAMENTE con el JSON, sin explicaciones ni markdown.

Ticket:
{ticket}

Extrae estos campos:
{{
  "nombre_cliente": "nombre completo o null",
  "categoria": "uno de: bug_tecnico | consulta_comercial | facturacion | cancelacion | sugerencia | acceso_cuenta",
  "urgencia": "uno de: alta | media | baja",
  "sentimiento": "uno de: muy_negativo | negativo | neutro | positivo | muy_positivo",
  "plan_actual": "básico | estándar | premium | enterprise | desconocido",
  "requiere_accion_inmediata": true o false,
  "resumen": "máximo 15 palabras"
}}
""")

chain_extraccion = prompt_extraccion | llm | StrOutputParser()


def extraer_datos(ticket_texto: str) -> dict:
    """
    Extrae datos estructurados de un ticket de texto libre usando LLM.
    Retorna un dict con los campos del JSON, o dict vacío si falla el parseo.
    """
    raw = chain_extraccion.invoke({"ticket": ticket_texto})
    # El LLM puede devolver el JSON envuelto en ```json ... ``` — hay que limpiarlo
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)


# Procesar todos los tickets en lote
print("Procesando tickets con el LLM...\n")
tickets_estructurados = []

for i, ticket in enumerate(tickets_texto):
    print(f"  Procesando ticket {i+1}/{len(tickets_texto)}...", end=" ")
    datos = extraer_datos(ticket)
    datos["texto_original"] = ticket  # guardamos el texto original para referencia
    tickets_estructurados.append(datos)
    print(f"✅ → categoría: {datos['categoria']}, urgencia: {datos['urgencia']}")

df_tickets = pd.DataFrame(tickets_estructurados)
print(f"\n✅ Extracción completa. DataFrame resultante:")
df_tickets[["nombre_cliente", "categoria", "urgencia", "sentimiento", "requiere_accion_inmediata", "resumen"]]

## PASO 6 — Análisis Exploratorio y Priorización con ML Downstream

### ¿Qué es el "ML downstream"?

Una vez que el LLM ha convertido el texto en datos estructurados, podemos aplicar 
cualquier técnica de ML o análisis que queramos. Aquí usamos un **score de prioridad** 
calculado como combinación ponderada de los campos extraídos.

Este es el poder del PATRÓN B: el LLM hace el trabajo duro de interpretación del texto,
y el ML hace el trabajo de cómputo rápido sobre los datos ya estructurados.

### LabelEncoder para múltiples columnas

Aplicamos `LabelEncoder` a cada columna categórica para convertirla en número.  
Los encoders se guardan en un dict `encoders` para poder aplicarlos a datos nuevos.

### El score de prioridad

```python
score = accion_inmediata * 40    # pesa más si requiere acción
      + urgencia * 20            # urgencia alta suma puntos
      + sentimiento * 15         # sentimiento muy negativo suma puntos
      + categoria * 5            # tipo de categoria como desempate
```

Nota: la urgencia "alta" tiene menor valor de encoding que "baja" (orden alfabético),
por eso se hace `3 - urgencia_enc` para invertir: alta urgencia = más puntos.

In [ ]:
from sklearn.preprocessing import LabelEncoder

print("=" * 50)
print("📊 ANÁLISIS DE LOS DATOS EXTRAÍDOS")
print("=" * 50)
print("\nDistribución por categoría:")
print(df_tickets["categoria"].value_counts().to_string())
print("\nDistribución por sentimiento:")
print(df_tickets["sentimiento"].value_counts().to_string())

# --- Priorización ML ---
df_ml = df_tickets.copy()

# Encodear todas las columnas categóricas
encoders = {}
for col in ["categoria", "sentimiento", "urgencia", "plan_actual"]:
    le = LabelEncoder()
    df_ml[f"{col}_enc"] = le.fit_transform(df_ml[col].fillna("desconocido"))
    encoders[col] = le

df_ml["requiere_accion_num"] = df_ml["requiere_accion_inmediata"].astype(int)

# Score de prioridad: combinación ponderada de factores de riesgo
df_ml["score_prioridad"] = (
    df_ml["requiere_accion_num"] * 40 +
    (3 - df_ml["urgencia_enc"]) * 20 +
    (2 - df_ml["sentimiento_enc"].clip(0, 2)) * 15 +
    df_ml["categoria_enc"] * 5
).astype(int)

# Clasificar en niveles de prioridad usando rangos
df_ml["nivel_prioridad"] = pd.cut(
    df_ml["score_prioridad"],
    bins=[-1, 20, 45, 200],
    labels=["🟢 Normal", "🟡 Prioritario", "🔴 Urgente"]
)

resultado = df_ml[["nombre_cliente", "categoria", "sentimiento", "urgencia",
                    "score_prioridad", "nivel_prioridad"]].sort_values("score_prioridad", ascending=False)

print("\n\n📋 COLA DE TICKETS PRIORIZADA:")
print("(Los primeros son los que el sistema atiende antes)")
print(resultado.to_string(index=False))

## PASO 8 — Generación Automática de Respuesta Inicial

### El pipeline completo: texto → extracción → priorización → respuesta

Aquí cerramos el ciclo. Para el ticket más urgente, el sistema genera automáticamente 
una respuesta inicial personalizada. Esto permite:

1. Respuesta inmediata al cliente (sin esperar a un agente humano)
2. El agente humano ve el ticket ya priorizado y con una respuesta borrador
3. La respuesta usa el sentimiento detectado para ajustar el tono

### ¿Por qué este segundo prompt también usa temperatura baja?

Las respuestas de soporte deben ser **profesionales y consistentes**, no creativas.
Un cliente frustrado no quiere sorpresas estilísticas, quiere sentirse escuchado.

In [ ]:
prompt_respuesta = ChatPromptTemplate.from_template("""
Eres un agente de soporte empático y profesional.
Genera un primer mensaje de respuesta para este ticket.

Cliente: {nombre_cliente}
Categoría: {categoria}
Sentimiento del cliente: {sentimiento}
Urgencia: {urgencia}
Resumen del problema: {resumen}
Prioridad asignada: {nivel_prioridad}

Genera una respuesta inicial de 3-4 oraciones que:
- Reconozca el problema específico del cliente por su nombre
- Sea empática según el sentimiento detectado (si es muy_negativo, mucha empatía)
- Indique el próximo paso concreto
- No prometa tiempos que no puedas cumplir
""")

chain_respuesta = prompt_respuesta | llm | StrOutputParser()

# Seleccionamos el ticket con mayor score de prioridad (el más urgente)
ticket_urgente = df_ml.loc[df_ml["score_prioridad"].idxmax()]

respuesta = chain_respuesta.invoke({
    "nombre_cliente": ticket_urgente["nombre_cliente"],
    "categoria": ticket_urgente["categoria"],
    "sentimiento": ticket_urgente["sentimiento"],
    "urgencia": ticket_urgente["urgencia"],
    "resumen": ticket_urgente["resumen"],
    "nivel_prioridad": str(ticket_urgente["nivel_prioridad"])
})

print(f"\n📧 RESPUESTA AUTOMÁTICA GENERADA")
print(f"Para: {ticket_urgente['nombre_cliente']} ({ticket_urgente['categoria']})")
print(f"Score de prioridad: {ticket_urgente['score_prioridad']}")
print("-" * 50)
print(respuesta)

## Conclusiones del Notebook T4.2

### Lo que has aprendido

1. **PATRÓN B** resuelve el problema de datos no estructurados:  
   `Texto libre → LLM (parser) → JSON → DataFrame → ML/análisis`

2. **Temperature=0** es obligatorio cuando necesitas que el LLM devuelva un formato exacto (JSON).
   Con temperatura alta, el LLM puede inventarse campos o cambiar el formato.

3. **Parser JSON robusto**: siempre limpia la respuesta del LLM antes de parsear —  
   los backticks de markdown son el error más común.

4. **Composición de chains**: puedes tener múltiples chains en un mismo pipeline  
   (extracción, priorización, respuesta). Cada una hace una tarea específica.

5. **ML downstream**: una vez que el LLM estructura los datos, cualquier técnica de ML  
   o análisis puede aplicarse sobre ese DataFrame como si fuera datos tabulares normales.

---

### ¿Cuándo usar este patrón en proyectos reales?

- Sistemas de ticketing de soporte al cliente
- Procesamiento de formularios de texto libre
- Extracción de entidades de contratos o documentos legales
- Normalización de datos provenientes de múltiples fuentes con formatos distintos
- ETL con datos semi-estructurados (emails, logs, notas)

---

### Patrón clave (para memorizar)

```
Texto libre → LLM (extracción JSON) → DataFrame estructurado → ML downstream → Acción
```